In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from sklearn.preprocessing import LabelEncoder, StandardScaler
import lightgbm as lgb
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import optuna
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [1]:
class Paths:
    p = "/Users/shirokoshikentaro/Desktop/Python3/house-prices-advanced-regression-techniques"
    train = p + "/train.csv"
    test = p + "/test.csv"
    sample = p + "/sample_submission.csv"

In [3]:
# ===== ステップ1: データ読み込み =====
print("=" * 60)
print("データ読み込み中...")
print("=" * 60)

train = pd.read_csv(Paths.train)
test = pd.read_csv(Paths.test)

print(f"train shape: {train.shape}")
print(f"test shape: {test.shape}")


データ読み込み中...
train shape: (1460, 81)
test shape: (1459, 80)


In [4]:
# より慎重な外れ値除去
outliers = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)].index
print(f"GrLivArea外れ値: {len(outliers)}件を除去")
train = train.drop(outliers)

GrLivArea外れ値: 2件を除去


In [5]:
# y_trainを再計算
y_train = np.log1p(train["SalePrice"].values)
print(f"y_train shape: {y_train.shape}")

y_train shape: (1458,)


In [6]:
# ===== ステップ2: 新規特徴量作成 =====
print("\n" + "=" * 60)
print("特徴量エンジニアリング中...")
print("=" * 60)

for df in [train, test]:
    # 交互作用特徴量
    df["QualGrLiv"] = df["OverallQual"] * df["GrLivArea"]
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["QualTotalSF"] = df["OverallQual"] * df["TotalSF"]
    
    # 年齢関連
    df["Age"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    
    # バスルーム集計
    df["TotalBath"] = (df["FullBath"] + 0.5*df["HalfBath"].fillna(0) + 
                       df["BsmtFullBath"].fillna(0) + 0.5*df["BsmtHalfBath"].fillna(0))
    
    # 面積系
    df["AreaPerRoom"] = df["GrLivArea"] / df["TotRmsAbvGrd"].replace(0, 1)
    df["GarageScore"] = df["GarageCars"] * df["GarageArea"]
    
    # 対数変換
    df["Log_GrLivArea"] = np.log1p(df["GrLivArea"])
    df["Log_LotArea"] = np.log1p(df["LotArea"])
    df["Log_TotalSF"] = np.log1p(df["TotalSF"])
    
    # その他
    df["QualGrLivLog"] = df["OverallQual"] * df["Log_GrLivArea"]
    df["TotalPorchSF"] = (df["OpenPorchSF"] + df["EnclosedPorch"] + 
                          df["3SsnPorch"].fillna(0) + df["ScreenPorch"].fillna(0))
    
    # フラグ特徴量
    df["IsNew"] = (df["Age"] <= 5).astype(int)
    df["IsRemodeled"] = (df["YearRemodAdd"] != df["YearBuilt"]).astype(int)
    df["HasBsmt"] = (df["TotalBsmtSF"] > 0).astype(int)
    df["BsmtScore"] = df["TotalBsmtSF"] * df["HasBsmt"]


特徴量エンジニアリング中...


In [8]:
# ===== ステップ3: 特徴量定義 =====
num_features = [
    "LotArea", "YearBuilt", "YearRemodAdd",
    "GrLivArea", "TotalBsmtSF", "1stFlrSF", "2ndFlrSF",
    "FullBath", "BedroomAbvGr", "TotRmsAbvGrd",
    "GarageCars", "GarageArea", "OverallQual", "OverallCond",
    "QualGrLiv", "TotalSF", "QualTotalSF", 
    "Age", "RemodAge", "TotalBath", "AreaPerRoom", "GarageScore",
    "WoodDeckSF", "OpenPorchSF", "EnclosedPorch",
    "Fireplaces", "HalfBath", "BsmtFullBath",
    "QualGrLivLog", "TotalPorchSF", "IsNew", "IsRemodeled", 
    "HasBsmt", "BsmtScore",
]

cat_features = [
    "Neighborhood", "BldgType", "HouseStyle",
    "MSZoning", "Foundation", "GarageType",
    "ExterQual", "KitchenQual", "BsmtQual",
    "HeatingQC", "FireplaceQu", "GarageFinish",
]

all_features = num_features + cat_features
print(f"総特徴量: {len(all_features)}個 (数値: {len(num_features)}, カテゴリ: {len(cat_features)})")

総特徴量: 46個 (数値: 34, カテゴリ: 12)


In [9]:
# ===== ステップ4: 欠損値処理 =====
print("\n欠損値処理中...")
for col in num_features:
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

for col in cat_features:
    train[col] = train[col].fillna("None")
    test[col] = test[col].fillna("None")


欠損値処理中...


In [ ]:
# ===== ステップ5: エンコーディング =====
print("カテゴリ変数のエンコーディング中...")
for col in cat_features:
    le = LabelEncoder()
    combined = pd.concat([train[col].astype(str), test[col].astype(str)])
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

カテゴリ変数のエンコーディング中...


In [11]:
# ===== ステップ6: X_train, X_test作成 =====
X_train = train[all_features].values
X_test = test[all_features].values

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (1458, 46)
X_test shape: (1459, 46)


In [15]:
# ===== ステップ7: Optunaによる最適化 =====
print("\n" + "=" * 60)
print("Optunaによるハイパーパラメータ最適化開始")
print("=" * 60)

def objective(trial):
    """Optunaの目的関数"""
    
    # === LightGBMのパラメータ提案 ===
    params_lgb = {
        "boosting_type": "gbdt",
        "objective": "regression",
        "metric": "rmse",
        "num_leaves": trial.suggest_int("num_leaves", 20, 60),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "n_estimators": 10000,
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 30),
        "subsample": trial.suggest_float("subsample", 0.6, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.95),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 10.0, log=True),
        "random_state": 123,
        "verbose": -1,
    }
    
    # === Ridge/Lassoのパラメータ提案 ===
    ridge_alpha = trial.suggest_float("ridge_alpha", 1, 100, log=True)
    lasso_alpha = trial.suggest_float("lasso_alpha", 0.0001, 0.01, log=True)
    
    # === アンサンブルの重み提案 ===
    w_lgb = trial.suggest_float("w_lgb", 0.4, 0.9)
    w_ridge = trial.suggest_float("w_ridge", 0.05, 0.4)
    w_lasso = 1.0 - w_lgb - w_ridge
    
    # 重みの合計が1を超える場合はスキップ
    if w_lasso < 0 or w_lasso > 0.5:
        return float('inf')
    
    # === 5分割交差検証 ===
    cv = KFold(n_splits=5, shuffle=True, random_state=123)
    lgb_oof = np.zeros(len(X_train))
    ridge_oof = np.zeros(len(X_train))
    lasso_oof = np.zeros(len(X_train))
    
    for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
        X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
        X_val, y_val = X_train[val_idx], y_train[val_idx]
        
        # LightGBM
        model_lgb = lgb.LGBMRegressor(**params_lgb)
        model_lgb.fit(
            X_tr, y_tr, 
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
        )
        lgb_oof[val_idx] = model_lgb.predict(X_val)
        
        # Ridge & Lasso (標準化が必要)
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_val_scaled = scaler.transform(X_val)
        
        model_ridge = Ridge(alpha=ridge_alpha, random_state=123)
        model_ridge.fit(X_tr_scaled, y_tr)
        ridge_oof[val_idx] = model_ridge.predict(X_val_scaled)
        
        model_lasso = Lasso(alpha=lasso_alpha, random_state=123, max_iter=5000)
        model_lasso.fit(X_tr_scaled, y_tr)
        lasso_oof[val_idx] = model_lasso.predict(X_val_scaled)
    
    # アンサンブル予測
    ensemble_oof = w_lgb * lgb_oof + w_ridge * ridge_oof + w_lasso * lasso_oof
    score = np.sqrt(mean_squared_error(y_train, ensemble_oof))
    
    return score

# Optunaスタディの実行
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10, show_progress_bar=True)

print("\n" + "=" * 60)
print("最適化完了！")
print("=" * 60)
print(f"Best CV Score: {study.best_value:.5f}")
print(f"\n最適パラメータ:")
for key, value in study.best_params.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")


Optunaによるハイパーパラメータ最適化開始


  0%|          | 0/10 [00:00<?, ?it/s]

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[252]	valid_0's rmse: 0.125188
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[365]	valid_0's rmse: 0.12794
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[331]	valid_0's rmse: 0.138897
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[314]	valid_0's rmse: 0.137993
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[192]	valid_0's rmse: 0.121767
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[2527]	valid_0's rmse: 0.138859
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[2160]	valid_0's rmse: 0.143798
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[3354]

In [16]:
# ===== ステップ8: 最適パラメータで最終モデル訓練 =====
print("\n" + "=" * 60)
print("最適パラメータで最終モデル訓練中...")
print("=" * 60)

best_params = study.best_params

# 最適パラメータの抽出
params_lgb_best = {
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": "rmse",
    "num_leaves": best_params["num_leaves"],
    "learning_rate": best_params["learning_rate"],
    "n_estimators": 10000,
    "min_child_samples": best_params["min_child_samples"],
    "subsample": best_params["subsample"],
    "colsample_bytree": best_params["colsample_bytree"],
    "reg_alpha": best_params["reg_alpha"],
    "reg_lambda": best_params["reg_lambda"],
    "random_state": 123,
    "verbose": -1,
}

ridge_alpha_best = best_params["ridge_alpha"]
lasso_alpha_best = best_params["lasso_alpha"]
w_lgb_best = best_params["w_lgb"]
w_ridge_best = best_params["w_ridge"]
w_lasso_best = 1.0 - w_lgb_best - w_ridge_best

print(f"\n[アンサンブル重み]")
print(f"LightGBM: {w_lgb_best:.3f}")
print(f"Ridge:    {w_ridge_best:.3f}")
print(f"Lasso:    {w_lasso_best:.3f}")


最適パラメータで最終モデル訓練中...

[アンサンブル重み]
LightGBM: 0.568
Ridge:    0.101
Lasso:    0.331


In [17]:
# 最終モデルの訓練
cv = KFold(n_splits=5, shuffle=True, random_state=123)

lgb_oof_final = np.zeros(len(X_train))
ridge_oof_final = np.zeros(len(X_train))
lasso_oof_final = np.zeros(len(X_train))

lgb_test_preds = []
ridge_test_preds = []
lasso_test_preds = []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_val, y_val = X_train[val_idx], y_train[val_idx]
    
    # LightGBM
    model_lgb = lgb.LGBMRegressor(**params_lgb_best)
    model_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )
    lgb_oof_final[val_idx] = model_lgb.predict(X_val)
    lgb_test_preds.append(model_lgb.predict(X_test))
    
    # Ridge & Lasso
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    model_ridge = Ridge(alpha=ridge_alpha_best, random_state=123)
    model_ridge.fit(X_tr_scaled, y_tr)
    ridge_oof_final[val_idx] = model_ridge.predict(X_val_scaled)
    ridge_test_preds.append(model_ridge.predict(X_test_scaled))
    
    model_lasso = Lasso(alpha=lasso_alpha_best, random_state=123, max_iter=5000)
    model_lasso.fit(X_tr_scaled, y_tr)
    lasso_oof_final[val_idx] = model_lasso.predict(X_val_scaled)
    lasso_test_preds.append(model_lasso.predict(X_test_scaled))
    
    print(f"Fold {fold+1}/5 完了")

# 各モデルの個別スコア
lgb_score = np.sqrt(mean_squared_error(y_train, lgb_oof_final))
ridge_score = np.sqrt(mean_squared_error(y_train, ridge_oof_final))
lasso_score = np.sqrt(mean_squared_error(y_train, lasso_oof_final))

print(f"\n[個別モデルのCVスコア]")
print(f"LightGBM: {lgb_score:.5f}")
print(f"Ridge:    {ridge_score:.5f}")
print(f"Lasso:    {lasso_score:.5f}")

# アンサンブルスコア
ensemble_oof = (w_lgb_best * lgb_oof_final + 
                w_ridge_best * ridge_oof_final + 
                w_lasso_best * lasso_oof_final)
ensemble_score = np.sqrt(mean_squared_error(y_train, ensemble_oof))

print(f"\n[アンサンブルCVスコア]")
print(f"最終スコア: {ensemble_score:.5f}")

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[220]	valid_0's rmse: 0.120127
Fold 1/5 完了
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[193]	valid_0's rmse: 0.130279
Fold 2/5 完了
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[200]	valid_0's rmse: 0.141937
Fold 3/5 完了
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[223]	valid_0's rmse: 0.13542
Fold 4/5 完了
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[191]	valid_0's rmse: 0.118962
Fold 5/5 完了

[個別モデルのCVスコア]
LightGBM: 0.12965
Ridge:    0.12734
Lasso:    0.12723

[アンサンブルCVスコア]
最終スコア: 0.12268


In [18]:
# ===== ステップ9: テストデータ予測 =====
print("\n" + "=" * 60)
print("テストデータ予測中...")
print("=" * 60)

lgb_test_pred = np.mean(lgb_test_preds, axis=0)
ridge_test_pred = np.mean(ridge_test_preds, axis=0)
lasso_test_pred = np.mean(lasso_test_preds, axis=0)

y_pred_log = (w_lgb_best * lgb_test_pred + 
              w_ridge_best * ridge_test_pred + 
              w_lasso_best * lasso_test_pred)
y_pred = np.expm1(y_pred_log)


テストデータ予測中...


In [20]:

# ===== ステップ10: 提出ファイル作成 =====
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": y_pred
})

output_file = "submission_optuna_ensemble.csv"
submission.to_csv(output_file, index=False)

print(f"\n✅ 保存完了: {output_file}")
print(f"📊 予測価格統計:")
print(f"   平均: ${y_pred.mean():,.0f}")
print(f"   中央値: ${np.median(y_pred):,.0f}")
print(f"   最小: ${y_pred.min():,.0f}")
print(f"   最大: ${y_pred.max():,.0f}")
print("\n" + "=" * 60)
print("全処理完了！")
print("=" * 60)


✅ 保存完了: submission_optuna_ensemble.csv
📊 予測価格統計:
   平均: $178,400
   中央値: $156,823
   最小: $56,581
   最大: $845,472

全処理完了！
